# 🗺️ TSP Data Generator

This notebook generates **training** and **validation** datasets for the **Travelling Salesman Problem (TSP)** using a **uniform distribution** over $[0, 1]^2$.

| Parameter | Value |
|---|---|
| Problem sizes | 10, 20, 30, …, 200 |
| Training instances per size | 128 000 |
| Validation instances per size | 10 000 |
| Point distribution | $\mathcal{U}(0, 1)^2$ |
| Storage format | NumPy `.npy` |
| Destination | Google Drive |

Each instance is a tensor of shape `(num_nodes, 2)` containing 2-D coordinates sampled i.i.d. from $\text{Uniform}(0,1)$.

---
## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 2 — Configuration

Adjust the parameters below if needed.

In [ ]:
import os
import time
import numpy as np

# ── Problem sizes ────────────────────────────────────────────────────────────
TSP_SIZES = list(range(10, 201, 10))  # [10, 20, 30, ..., 200]

# ── Dataset sizes ────────────────────────────────────────────────────────────
NUM_TRAIN = 128_000
NUM_VAL   = 10_000

# ── Random seed (set to None for non-reproducible) ────────────────────────────
SEED = 12345

# ── Google Drive output directory ─────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/TSP_Data'

# ── Chunk size for large problem sizes (avoids OOM on Colab) ──────────────────
# Instances are generated in chunks and concatenated.
# Decrease this if you hit memory errors on larger TSP sizes.
CHUNK_SIZE = 32_000

print(f"TSP sizes : {TSP_SIZES}")
print(f"Train / Val: {NUM_TRAIN:,} / {NUM_VAL:,}")
print(f"Seed       : {SEED}")
print(f"Output dir : {DRIVE_ROOT}")

---
## 3 — Helper Functions

In [ ]:
def generate_tsp_instances(
    rng: np.random.Generator,
    num_instances: int,
    num_nodes: int,
    chunk_size: int = 32_000,
) -> np.ndarray:
    """
    Generate TSP instances by sampling 2-D coordinates from U(0, 1).

    To stay within Colab's RAM limits for large problem sizes,
    instances are generated in chunks and concatenated.

    Parameters
    ----------
    rng : np.random.Generator
        NumPy random generator (for reproducibility).
    num_instances : int
        Total number of instances to generate.
    num_nodes : int
        Number of cities (nodes) per TSP instance.
    chunk_size : int
        Number of instances generated per chunk.

    Returns
    -------
    np.ndarray  shape (num_instances, num_nodes, 2), dtype float32
    """
    chunks = []
    remaining = num_instances
    while remaining > 0:
        batch = min(remaining, chunk_size)
        chunk = rng.random((batch, num_nodes, 2), dtype=np.float32)
        chunks.append(chunk)
        remaining -= batch
    return np.concatenate(chunks, axis=0)


def save_dataset(data: np.ndarray, filepath: str) -> None:
    """Save a NumPy array to disk and print file size."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    np.save(filepath, data)
    size_mb = os.path.getsize(filepath) / (1024 ** 2)
    print(f"  ✅  Saved {filepath}  ({size_mb:.1f} MB)")


def format_time(seconds: float) -> str:
    """Human-readable elapsed time."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = int(seconds // 60)
    secs = seconds % 60
    return f"{minutes}m {secs:.1f}s"

---
## 4 — Generate & Save Data

Each problem size gets its own subdirectory:

```
Google Drive / MyDrive / TSP_Data /
├── tsp_10/
│   ├── train.npy      (128000, 10, 2)
│   └── val.npy        (10000, 10, 2)
├── tsp_20/
│   ├── train.npy      (128000, 20, 2)
│   └── val.npy        (10000, 20, 2)
├── ...
└── tsp_200/
    ├── train.npy      (128000, 200, 2)
    └── val.npy        (10000, 200, 2)
```

In [ ]:
rng = np.random.default_rng(SEED)
total_start = time.time()

summary_rows = []  # for the summary table

for n in TSP_SIZES:
    print(f"\n{'═' * 60}")
    print(f"  TSP-{n}")
    print(f"{'═' * 60}")

    out_dir = os.path.join(DRIVE_ROOT, f'tsp_{n}')

    # ── Train ──
    t0 = time.time()
    train_data = generate_tsp_instances(rng, NUM_TRAIN, n, CHUNK_SIZE)
    gen_time = time.time() - t0
    print(f"  Generated {NUM_TRAIN:,} train instances  "
          f"shape={train_data.shape}  ({format_time(gen_time)})")
    save_dataset(train_data, os.path.join(out_dir, 'train.npy'))
    train_mb = os.path.getsize(os.path.join(out_dir, 'train.npy')) / (1024**2)

    # ── Validation ──
    t0 = time.time()
    val_data = generate_tsp_instances(rng, NUM_VAL, n, CHUNK_SIZE)
    gen_time = time.time() - t0
    print(f"  Generated {NUM_VAL:,}  val  instances  "
          f"shape={val_data.shape}  ({format_time(gen_time)})")
    save_dataset(val_data, os.path.join(out_dir, 'val.npy'))
    val_mb = os.path.getsize(os.path.join(out_dir, 'val.npy')) / (1024**2)

    summary_rows.append((n, train_data.shape, val_data.shape,
                         train_mb, val_mb))

    # Free memory
    del train_data, val_data

total_elapsed = time.time() - total_start
print(f"\n{'━' * 60}")
print(f"  🎉  All done in {format_time(total_elapsed)}")
print(f"{'━' * 60}")

---
## 5 — Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(summary_rows,
                  columns=['TSP Size', 'Train Shape', 'Val Shape',
                           'Train (MB)', 'Val (MB)'])
df['Total (MB)'] = df['Train (MB)'] + df['Val (MB)']
df['Train (MB)'] = df['Train (MB)'].map('{:.1f}'.format)
df['Val (MB)']   = df['Val (MB)'].map('{:.1f}'.format)
df['Total (MB)'] = df['Total (MB)'].map('{:.1f}'.format)

print(f"\nTotal storage: "
      f"{sum(float(x) for x in df['Total (MB)']):.0f} MB\n")
df

---
## 6 — Quick Sanity Check

Load back one file and verify shape, dtype, and value range.

In [ ]:
# Pick a random TSP size to verify
check_n = 50
check_path = os.path.join(DRIVE_ROOT, f'tsp_{check_n}', 'train.npy')

if os.path.exists(check_path):
    loaded = np.load(check_path)
    print(f"Loaded: {check_path}")
    print(f"  Shape : {loaded.shape}")
    print(f"  Dtype : {loaded.dtype}")
    print(f"  Min   : {loaded.min():.6f}")
    print(f"  Max   : {loaded.max():.6f}")
    print(f"  Mean  : {loaded.mean():.6f}  (expected ≈ 0.5)")
    assert loaded.shape == (NUM_TRAIN, check_n, 2)
    assert loaded.dtype == np.float32
    assert 0.0 <= loaded.min() and loaded.max() <= 1.0
    print("  ✅  All assertions passed!")
else:
    print(f"⚠️  {check_path} not found — did TSP-{check_n} get generated?")

---
## 7 — Visualise a Random Instance

In [ ]:
import matplotlib.pyplot as plt

viz_n = 50
viz_path = os.path.join(DRIVE_ROOT, f'tsp_{viz_n}', 'val.npy')

if os.path.exists(viz_path):
    viz_data = np.load(viz_path)
    idx = np.random.randint(len(viz_data))
    instance = viz_data[idx]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(instance[:, 0], instance[:, 1],
               c='#4A90D9', s=60, edgecolors='white', linewidths=0.8,
               zorder=3)
    for i, (x, y) in enumerate(instance):
        ax.annotate(str(i), (x, y), fontsize=7, ha='center', va='bottom',
                    xytext=(0, 5), textcoords='offset points', color='#333')

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_aspect('equal')
    ax.set_title(f'TSP-{viz_n}  —  Instance #{idx}', fontsize=14)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️  {viz_path} not found.")

---
## 📁 Output Structure

After running, you will find the following in your Google Drive:

```
MyDrive/TSP_Data/
├── tsp_10/   train.npy (128000, 10, 2)   val.npy (10000, 10, 2)
├── tsp_20/   train.npy (128000, 20, 2)   val.npy (10000, 20, 2)
├── tsp_30/   train.npy (128000, 30, 2)   val.npy (10000, 30, 2)
│   ...
├── tsp_190/  train.npy (128000, 190, 2)  val.npy (10000, 190, 2)
└── tsp_200/  train.npy (128000, 200, 2)  val.npy (10000, 200, 2)
```

### Loading data in your training script

```python
import numpy as np

train = np.load('/content/drive/MyDrive/TSP_Data/tsp_50/train.npy')  # (128000, 50, 2)
val   = np.load('/content/drive/MyDrive/TSP_Data/tsp_50/val.npy')    # (10000, 50, 2)
```